# Method 3: kNN-VC (voice conversion)

**Nothing is trained in this notebook, and this method does not read the text directly.** It first makes normal speech for your 20 sentences with a generic voice (edge-tts, the same one your chatbot uses), then converts that speech into the teacher's voice using recordings of the teacher.

## What you upload
- **One or more audio/video files of the teacher** (mp3, wav, m4a or mp4). You can select several files at once.
- Only the teacher's voice: no music, no students, no other voices.
- **More is better, up to about 5 minutes in total.** Quality gains taper off after about 5 minutes (per the kNN-VC README). A single 15 second clip works but sounds worse.
- **No transcript needed** for this method.
- **The 16 kHz wav conversion is done for you by Step 3.** You do not convert anything. (If you ever want to do it by hand, the command is in the box below Step 3.)

## What you get
- 20 audio files `00.wav` to `19.wav` and a zip `knnvc_results.zip` that downloads at the end. Keep it for the compare notebook.

## Steps
1. `Runtime > Change runtime type > T4 GPU`.
2. Run the cells from top to bottom. Step 3 asks you to upload the files.
3. Needs internet access for edge-tts (Colab has it).

## Step 1: Install (run this first)

In [1]:
!pip -q install edge-tts soundfile


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


## Step 2: Check the GPU and edit the config

In [3]:
import torch
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (it will still run on CPU, just slower)")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# ===================== CONFIG: edit only this cell =====================
MAX_MINUTES = 5        # use at most this many minutes of teacher audio

# The 20 sentences below are the same in all three method notebooks, so the results can be compared. If you change them in one notebook, change them identically in the others.
SENTENCES = [
    "Good morning everyone, let us begin with a quick recap of the previous lecture.",
    "Please open your notebooks and follow along with the example on the screen.",
    "Today we will look at how the method works and why it matters in practice.",
    "If you have any doubts, please ask them at the end of this section.",
    "The assignment is due next week, so please start working on it early.",
    "Let us go through the main steps one by one.",
    "This is a common mistake that students make during the exam.",
    "Remember that the goal is to understand the idea, not just memorize the formula.",
    "We will now move on to the next topic.",
    "To summarize, there are three key points you should take away from this lecture.",
    "A convolution slides a small kernel across the image to produce a feature map.",
    "Histogram equalization improves the contrast of an image.",
    "The Canny edge detector uses two thresholds for hysteresis.",
    "Backpropagation computes gradients using the chain rule.",
    "Gradient descent updates the weights in the direction that reduces the loss.",
    "A convolutional neural network applies pooling to reduce the spatial resolution.",
    "The ResNet architecture uses skip connections to ease the training of deep networks.",
    "Overfitting happens when the model memorizes the training data.",
    "Precision and recall describe different kinds of errors in a classifier.",
    "SIFT detects keypoints that are invariant to scale and rotation.",
]
# --- end of config ---

## Step 3: Upload the teacher files (converted to 16 kHz wav clips automatically)

In [ ]:
import os, sys, shutil, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("Command failed: " + cmd)
    return r.stdout

import glob, soundfile as sf
METHOD = "knnvc"
WORK = "/content/knnvc"
OUT = f"{WORK}/outputs"
SRC_DIR = f"{WORK}/source"
CLIP_DIR = f"{WORK}/teacher_16k"       # the 16 kHz wav clips end up here
for d in (OUT, SRC_DIR, CLIP_DIR):
    os.makedirs(d, exist_ok=True)

from google.colab import files
print("Click 'Choose files' and select your teacher recording(s). You can pick several.")
up = files.upload()

for k, name in enumerate(up):
    p = f"{SRC_DIR}/{k:02d}{os.path.splitext(name)[1]}"
    shutil.move(name, p)
    full = f"{WORK}/full_{k:02d}.wav"
    # convert anything to mono 16 kHz wav, then cut into 10 second clips
    sh(f'ffmpeg -y -loglevel error -i "{p}" -ac 1 -ar 16000 "{full}"')
    sh(f'ffmpeg -y -loglevel error -i "{full}" -f segment -segment_time 10 -c copy "{CLIP_DIR}/f{k:02d}_%03d.wav"')
    os.remove(full)

# drop tiny leftover pieces and respect MAX_MINUTES
total = 0.0
for p in sorted(glob.glob(f"{CLIP_DIR}/*.wav")):
    d = sf.info(p).duration
    if d < 2.0 or total + d > MAX_MINUTES * 60:
        os.remove(p)
    else:
        total += d
clips = sorted(glob.glob(f"{CLIP_DIR}/*.wav"))
print(f"{len(clips)} clips, {total / 60:.1f} minutes of teacher audio ready in {CLIP_DIR}")
if total < 120:
    print("WARNING: under 2 minutes. kNN-VC will work but sound weaker. More clean teacher audio helps.")
TEACHER_FILES_FOR_ZIP = clips[:20]

**By hand, if you ever want to** (one file to a 16 kHz mono wav):
```
ffmpeg -i your_file.mp3 -ac 1 -ar 16000 teacher_16k.wav
```

## Step 4: Make the source speech and convert it to the teacher's voice

In [ ]:
import torch, soundfile as sf

SRC_SPEECH = f"{WORK}/source_speech"
os.makedirs(SRC_SPEECH, exist_ok=True)

# 1) ordinary speech for each sentence in a generic voice, as 16 kHz wav
for i, s in enumerate(SENTENCES):
    mp3, wav = f"{SRC_SPEECH}/{i:02d}.mp3", f"{SRC_SPEECH}/{i:02d}.wav"
    subprocess.run([sys.executable, "-m", "edge_tts", "--voice", "en-US-AriaNeural", "--text", s, "--write-media", mp3], check=True)
    sh(f'ffmpeg -y -loglevel error -i {mp3} -ac 1 -ar 16000 {wav}')

# 2) convert it to the teacher's voice
def load16k(path):
    data, sr = sf.read(path, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    assert sr == 16000, f"{path} is {sr} Hz, expected 16000"
    return torch.from_numpy(data)

device = "cuda" if torch.cuda.is_available() else "cpu"
knn_vc = torch.hub.load("bshall/knn-vc", "knn_vc", prematched=True, trust_repo=True, pretrained=True, device=device)
matching_set = knn_vc.get_matching_set([load16k(p) for p in clips])
for i in range(len(SENTENCES)):
    query = knn_vc.get_features(load16k(f"{SRC_SPEECH}/{i:02d}.wav"))
    w = knn_vc.match(query, matching_set, topk=4)
    sf.write(f"{OUT}/{i:02d}.wav", w.detach().cpu().numpy().reshape(-1), 16000)
    print(f"{i + 1}/{len(SENTENCES)}")

## Step 5: Listen and download the results zip

In [ ]:
from IPython.display import Audio, display
for idx in (0, 13):          # one general sentence and one technical sentence
    p = f"{OUT}/{idx:02d}.wav"
    if os.path.exists(p):
        print(f"--- sentence {idx}: {SENTENCES[idx]}")
        display(Audio(p))

RES = f"/content/{METHOD}_results"
shutil.rmtree(RES, ignore_errors=True)
os.makedirs(f"{RES}/teacher_audio", exist_ok=True)
shutil.copytree(OUT, f"{RES}/outputs")
for p in TEACHER_FILES_FOR_ZIP:
    shutil.copy(p, f"{RES}/teacher_audio/")
open(f"{RES}/sentences.txt", "w").write("\n".join(SENTENCES) + "\n")
shutil.make_archive(f"/content/{METHOD}_results", "zip", RES)
from google.colab import files
files.download(f"/content/{METHOD}_results.zip")
print("Downloaded", f"{METHOD}_results.zip", "- keep it, the compare notebook needs it.")